# NLP Preprocessing Pipeline Assignment
Name: Sai Kadiravan S

Task 1:

1.  What is the difference between "Love" and "love" in NLP?

In most coding environments, "L" and "l" have different ASCII values. So, if you don't lowercase your text, the model sees them as two totally different words. It's like having two different entries in a ledger for the same person just because one was written in caps. We lowercase everything to keep the vocabulary small and clean. Lowercasing ensures consistency and reduces vocabulary size.

2. What happens if stopwords are not removed?

Words like "the", "is", and "at" are everywhere. If you're trying to figure out the topic of an article, these words are useless because they appear in every single document. If you don't remove them, the model gets confused by the "noise" and might fail to pick up the actual keywords like "Cricket" or "Himalayas".

3. Provide two real-world scenarios where removing stopwords can be harmful.

You can't just delete stopwords everywhere.
Scenario 1: If you're building a sentiment analysis tool, the word "not" is technically a stopword. But if you remove it from "not good", the machine just sees "good". Suddenly, a complaint looks like a compliment.

Scenario 2: Similarly, in Question Answering, words like "who" or "where" define what the user is actually looking for.

4. Explain the difference between stemming and lemmatization.

Stemming: It's a brute force approach that just chops off the end of a word based on fixed rules. It doesn't care about grammar or if the result is a real word.Example: "Caring" = "Car"
Result: Fast, but can change the entire meaning of the word.

Lemmatization: It's a smart process that uses a dictionary to find the actual root (the lemma). It understands the context and part of speech.
Example: "Better" = "Good"
Result: Accurate and meaningful, though it takes a bit more processing power.




In [24]:
!pip install nltk emoji

In [5]:
import re
import nltk
import emoji
from collections import Counter
import numpy as np

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [8]:
# Task 2
def preprocess_text(text):
    if not text or not isinstance(text, str):
        return [], ""

    # Remove URLs and emails
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Handle repeated characters (soooo becomes soo or so)
    text = re.sub(r'(.)\1+', r'\1', text)

    # Lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = text.split()

    # Remove short words (<=2), keep "no", "not"
    tokens = [word for word in tokens if len(word) > 2 or word in ['no', 'not']]

    # Final cleaned sentence
    cleaned_sentence = " ".join(tokens)

    return tokens, cleaned_sentence

In [11]:
# Task 3
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for text in test_sentences:
    tokens, clean_sent = preprocess_text(text)

    print("Original:", text)
    print("Tokens:", tokens)
    print("Cleaned:", clean_sent)#This verifies if punctuation and special chars are gone
    print("-" * 50)

    results.append((text, tokens, clean_sent))

Original: Get 100% FREE access now!!!
Tokens: ['get', 'fre', 'aces', 'now']
Cleaned: get fre aces now
--------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'loved', 'this', 'product']
Cleaned: absolutely loved this product
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['cal']
Cleaned: cal
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned: this the best course
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now
--------------------------------------------------
Original: Nooooo this is baaad!!!
Tokens: ['no', 'this', 'bad']
Cleaned: no this bad
-------------

In [13]:
# Task 4
import numpy as np

for text, tokens, clean_sent in results:
    if len(tokens) == 0:
        avg_len = 0
    else:
        avg_len = np.mean([len(word) for word in tokens])

    print("Sentence:", text)
    print("Total Tokens:", len(tokens))
    print("Unique Tokens:", len(set(tokens))) # Helpful for spotting repetition
    print("Avg Token Length:", round(avg_len, 2))
    print("-" * 50)

Sentence: Get 100% FREE access now!!!
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 3.25
--------------------------------------------------
Sentence: I absolutely looooved this product 😍😍
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 6.5
--------------------------------------------------
Sentence: Worst service ever... 0/10
Total Tokens: 3
Unique Tokens: 3
Avg Token Length: 5.33
--------------------------------------------------
Sentence: Call me at 9876543210
Total Tokens: 1
Unique Tokens: 1
Avg Token Length: 3.0
--------------------------------------------------
Sentence: This is THE best course!!!
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 4.25
--------------------------------------------------
Sentence: Visit https://openai.com now!
Total Tokens: 2
Unique Tokens: 2
Avg Token Length: 4.0
--------------------------------------------------
Sentence: Nooooo this is baaad!!!
Total Tokens: 3
Unique Tokens: 3
Avg Token Length: 3.0
----------------------------------------

In [16]:
# Task 5
all_tokens = []

for _, tokens, _ in results:
    all_tokens.extend(tokens)

freq = Counter(all_tokens)

print("Top 10 Frequent Words:")
print(freq.most_common(10))

print("\nTop 5 Least Frequent Words:")
print(freq.most_common()[-5:])

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('fre', 1), ('aces', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('limited', 1), ('ofer', 1), ('not', 1), ('hapy', 1), ('with', 1)]


In [22]:
# Task 6
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean_sent = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean_sent)  # Storing the cleaned sentences

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }
output = full_pipeline(test_sentences)
print(output)

{'tokens': ['get', 'fre', 'aces', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'cal', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'ofer', 'not', 'hapy', 'with', 'this'], 'clean_sentences': ['get fre aces now', 'absolutely loved this product', 'worst service ever', 'cal', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited ofer', 'not hapy with this']}


In [23]:
# Task 7
edge_cases = [
    "",
    "😂😂😂😂",
    "1234567890"
]

for case in edge_cases:
    tokens, clean = preprocess_text(case)
    print("Input:", case)
    print("Tokens:", tokens)
    print("Cleaned:", clean)
    print("-" * 40)

Input: 
Tokens: []
Cleaned: 
----------------------------------------
Input: 😂😂😂😂
Tokens: []
Cleaned: 
----------------------------------------
Input: 1234567890
Tokens: []
Cleaned: 
----------------------------------------
